In [ ]:
"""
Script 4: Trend Analysis
Analyze price trends, moving averages, and trend direction
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path

def calculate_daily_returns(df):
    """Calculate daily returns"""
    df['Daily_Return'] = df['Close Price'].pct_change() * 100
    df['Daily_Return_Abs'] = df['Close Price'].diff()
    
    return df

def trend_direction(df):
    """Determine trend direction (uptrend/downtrend)"""
    print("\n" + "="*70)
    print("TREND DIRECTION ANALYSIS")
    print("="*70)
    
    # Linear regression for trend
    x = np.arange(len(df))
    y = df['Close Price'].values
    slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(x, y)
    
    print(f"\nOverall Trend Analysis:")
    print(f"  Slope: {slope:.6f} (price change per day)")
    print(f"  R-squared: {r_value**2:.4f} (trend strength)")
    print(f"  P-value: {p_value:.6e} (statistical significance)")
    
    if slope > 0:
        trend = "UPTREND"
        print(f"  Direction: {trend} ↑")
    else:
        trend = "DOWNTREND"
        print(f"  Direction: {trend} ↓")
    
    return slope, r_value, trend

def analyze_moving_averages(df):
    """Analyze moving averages"""
    print("\n" + "="*70)
    print("MOVING AVERAGES ANALYSIS")
    print("="*70)
    
    # Summary statistics for moving averages
    print("\n3-Day Moving Average:")
    ma3 = df['3 Day MV.'].dropna()
    print(f"  Count: {len(ma3)}")
    print(f"  Mean: {ma3.mean():.4f}")
    print(f"  Min: {ma3.min():.4f}")
    print(f"  Max: {ma3.max():.4f}")
    print(f"  Std Dev: {ma3.std():.4f}")
    
    print("\n5-Day Moving Average:")
    ma5 = df['5 Day MV.'].dropna()
    print(f"  Count: {len(ma5)}")
    print(f"  Mean: {ma5.mean():.4f}")
    print(f"  Min: {ma5.min():.4f}")
    print(f"  Max: {ma5.max():.4f}")
    print(f"  Std Dev: {ma5.std():.4f}")
    
    # MA crossover signals
    print("\n3-Day vs 5-Day Moving Average:")
    df['MA_Diff'] = df['3 Day MV.'] - df['5 Day MV.']
    
    positive_diff = (df['MA_Diff'] > 0).sum()
    negative_diff = (df['MA_Diff'] < 0).sum()
    
    print(f"  3-Day MA > 5-Day MA: {positive_diff} days (Bullish)")
    print(f"  3-Day MA < 5-Day MA: {negative_diff} days (Bearish)")

def analyze_volatility(df):
    """Analyze price volatility"""
    print("\n" + "="*70)
    print("VOLATILITY ANALYSIS")
    print("="*70)
    
    # Calculate rolling volatility
    df['Volatility_20'] = df['Close Price'].rolling(window=20).std()
    df['Volatility_50'] = df['Close Price'].rolling(window=50).std()
    
    returns = df['Daily_Return'].dropna()
    
    print("\nDaily Returns Statistics:")
    print(f"  Mean Return: {returns.mean():.4f}%")
    print(f"  Std Dev (Volatility): {returns.std():.4f}%")
    print(f"  Min Return: {returns.min():.4f}%")
    print(f"  Max Return: {returns.max():.4f}%")
    print(f"  Sharpe Ratio (assuming 0% risk-free rate): {returns.mean() / returns.std():.4f}")
    
    # Volatility trends
    vol_20 = df['Volatility_20'].dropna()
    vol_50 = df['Volatility_50'].dropna()
    
    if len(vol_20) > 0:
        print(f"\n20-Day Rolling Volatility:")
        print(f"  Mean: {vol_20.mean():.4f}")
        print(f"  Current: {vol_20.iloc[-1]:.4f}")
        print(f"  Min: {vol_20.min():.4f}")
        print(f"  Max: {vol_20.max():.4f}")
    
    if len(vol_50) > 0:
        print(f"\n50-Day Rolling Volatility:")
        print(f"  Mean: {vol_50.mean():.4f}")
        print(f"  Current: {vol_50.iloc[-1]:.4f}")
        print(f"  Min: {vol_50.min():.4f}")
        print(f"  Max: {vol_50.max():.4f}")

def price_momentum(df):
    """Analyze price momentum"""
    print("\n" + "="*70)
    print("MOMENTUM ANALYSIS")
    print("="*70)
    
    # Calculate momentum (price change over periods)
    df['Momentum_5'] = df['Close Price'].diff(5)
    df['Momentum_10'] = df['Close Price'].diff(10)
    
    # Positive momentum days
    pos_momentum_5 = (df['Momentum_5'] > 0).sum()
    pos_momentum_10 = (df['Momentum_10'] > 0).sum()
    
    print(f"\n5-Day Momentum:")
    print(f"  Positive momentum days: {pos_momentum_5}")
    print(f"  Negative momentum days: {(df['Momentum_5'] < 0).sum()}")
    
    print(f"\n10-Day Momentum:")
    print(f"  Positive momentum days: {pos_momentum_10}")
    print(f"  Negative momentum days: {(df['Momentum_10'] < 0).sum()}")

def generate_trend_report(df):
    """Generate comprehensive trend report"""
    print("\n" + "="*70)
    print("TREND ANALYSIS REPORT")
    print("="*70)
    
    # Calculate daily returns
    df = calculate_daily_returns(df)
    
    # Trend direction
    slope, r_squared, trend = trend_direction(df)
    
    # Moving averages
    analyze_moving_averages(df)
    
    # Volatility
    analyze_volatility(df)
    
    # Momentum
    price_momentum(df)
    
    # Summary
    print("\n" + "="*70)
    print("TREND SUMMARY")
    print("="*70)
    price_range = df['Close Price'].max() - df['Close Price'].min()
    total_return = ((df['Close Price'].iloc[-1] - df['Close Price'].iloc[0]) / df['Close Price'].iloc[0]) * 100
    
    print(f"\nPrice Range: {df['Close Price'].min():.2f} to {df['Close Price'].max():.2f}")
    print(f"Total Price Change: {price_range:.2f} ({total_return:.2f}%)")
    print(f"Overall Trend: {trend}")
    print(f"Trend Strength (R²): {r_squared**2:.4f}")
    
    return df

if __name__ == "__main__":
    file_path = 'outputs/cleaned_data.csv'
    
    try:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        
        df = generate_trend_report(df)
        
        # Save enhanced dataset
        Path('outputs').mkdir(exist_ok=True)
        df.to_csv('outputs/data_with_indicators.csv', index=False)
        print(f"\n✓ Enhanced data saved to outputs/data_with_indicators.csv")
        print("✓ Trend analysis completed!")
        
    except FileNotFoundError:
        print(f"Please run 01_data_loading.py first to generate {file_path}")